# 策略熵骤降
- 正常LLM约 8-12 bits，若连续低于 5 bits 说明熵坍塌

## 策略熵是什么？
策略熵衡量的是模型在生成每个 token 时，候选词的概率分布有多"分散"。
  - 熵高 → <font color='red'>概率分布平坦</font> → 模型对多个候选词"犹豫不决" → <font color='red'>输出多样</font>
  - 熵低 → 概率分布尖锐 → 模型对某个词"极度确定" → 输出单一
  
### 公式为： 
$H=−∑p_ilog_2p_i$ ，单位是 bits。

### 具体例子
假设模型要生成下一个词，候选词有 4 个：{"今天", "明天", "昨天", "后天"}。

#### 正常情况（熵 ≈ 2.0 bits）
| 候选词 | 概率 |
|--------|------|
| 今天 | 0.30 |
| 明天 | 0.30 |
| 昨天 | 0.20 |
| 后天 | 0.20 |

<font color='red'>概率分布比较平坦，模型"犹豫"，每次采样结果不同。</font>

$H=−(0.3log_20.3×2+0.2log_20.2×2)≈1.97 bits$

#### 轻微偏低（熵 ≈ 1.0 bits）
| 候选词 | 概率 |
|--------|------|
| 今天 | 0.60 |
| 明天 | 0.20 |
| 昨天 | 0.10 |
| 后天 | 0.10 |

<font color='red'>模型开始倾向于"今天"，但仍有随机性。</font>

#### 熵坍塌（熵 ≈ 0.2 bits）

| 候选词 | 概率 |
|--------|------|
| 今天 | 0.95 |
| 明天 | 0.02 |
| 昨天 | 0.02 |
| 后天 | 0.01 |

<font color='red'>模型几乎 100% 确定选"今天"，输出变得高度确定、缺乏多样性。</font>
$H=−(0.95log_20.95+0.02log_20.02×2+0.01log_20.01)≈0.25 bits$

## 为什么"8-12 bits"是正常的？
实际 LLM 的词表通常有 5 万~10 万个 token。在正常对话中，模型面对的是开放式生成任务，候选词极多，概率分布相对分散。
- 如果词表有 50000 个词，完全均匀分布的熵为 $log_2{50000}≈15.6 bits。$
- <font color='red'>实际生成时，模型会排除大量不合理的词，概率集中在几百个候选词上，但仍有相当的不确定性。</font>
- 因此正常 LLM 的策略熵通常在 8-12 bits 之间。

## 为什么低于 5 bits 说明出了问题？
当策略熵连续低于 5 bits，<font color='red'>意味着模型在几乎每一步都把 99% 以上的概率集中在极少数 token 上</font>

实际表现就是：
- <font color='red'>输入不同问题</font> → <font color='blue'>输出几乎一样的固定句式</font>
- 采样温度再高也没用，因为概率分布本身已经"塌缩"
- 模型变成了"复读机"

### 直觉类比：
- 正常聊天时你有很多话可以说（高熵），但模式崩溃后你脑子里只剩一句话，不管别人问什么，你都说同一句（低熵）。

## 实操中的监控建议

| 指标 | 正常范围 | 预警 | 危险 |
|------|----------|------|------|
| 策略熵 | 8-12 bits | 5-8 bits | < 5 bits |
| 连续低熵步数 | — | > 10 步 | > 50 步 |

<font color='blue'>建议在训练日志中实时监控每步的平均策略熵，一旦连续 10 步低于 5 bits，立即触发预警并考虑增大熵正则化系数或回滚 checkpoint</font>。